### Import the Libraries

In [ ]:
import os
import pandas as pd
from IPython.display import display
from collections import namedtuple
from train import main

### Define the Experimental Grid

In [ ]:
seeds = [42, 123, 2026]
covariate_options = ['none', 'feature_d', 'binary', 'binary_and_d', 'all']
Args = namedtuple('Args', ['seed', 'cov_config', 'epochs', 'batch_size', 'learning_rate', 'patience', 'd_model', 'n_heads', 'e_layers'])

### Run the Ablation Study

In [ ]:
for cov_flag in covariate_options:
    for current_seed in seeds:
        print(f"\n{'='*50}")
        print(f"RUNNING EXPERIMENT | Seed: {current_seed} | Config: {cov_flag}")
        print(f"{'='*50}")
        
        args = Args(
            seed=current_seed,
            cov_config=cov_flag,
            epochs=15,          
            batch_size=32,
            learning_rate=1e-3,
            patience=3,
            d_model=64,         
            n_heads=4,
            e_layers=2
        )
        
        main(args)

### Analyze the Results

In [ ]:
results = {
    "Configuration": [
        "none", "none", "none",
        "feature_d", "feature_d", "feature_d",
        "binary", "binary", "binary",
        "binary_and_d", "binary_and_d", "binary_and_d",
        "all", "all", "all"
    ],
    "Seed": [
        42, 123, 2026,
        42, 123, 2026,
        42, 123, 2026,
        42, 123, 2026,
        42, 123, 2026
    ],
    "Val RMSE": [
        0.000, 0.000, 0.000,  # none
        0.000, 0.000, 0.000,  # feature_d
        0.000, 0.000, 0.000,  # binary
        0.000, 0.000, 0.000,  # binary_and_d
        0.000, 0.000, 0.000   # all
    ],
    "Parameters (P)": [
        0, 0, 0,  # none
        0, 0, 0,  # feature_d
        0, 0, 0,  # binary
        0, 0, 0,  # binary_and_d
        0, 0, 0   # all
    ]
}

df_raw = pd.DataFrame(results)

df_summary = df_raw.groupby("Configuration").agg({
    "Val RMSE": ["mean", "std"],
    "Parameters (P)": "first"  # Parameter count is static per configuration
}).round(4)

# Flatten multi-level columns for a cleaner display
df_summary.columns = ['Mean RMSE', 'RMSE Std Dev', 'Parameters (P)']

# Sort by lowest Mean RMSE to immediately identify the winning model
df_summary = df_summary.sort_values(by="Mean RMSE")

print("--- RAW SEED LOGS ---")
display(df_raw)

print("\n--- ABLATION SUMMARY (For PDF Report) ---")
display(df_summary)

## Automated Covariate Ablation
The cell below will automatically train the model on all covariate configurations and compile the best validation metrics into a single Pandas DataFrame.

In [ ]:
import subprocess
import json
import pandas as pd
import os

configs = ['none', 'binary', 'feature_d', 'binary_and_d', 'all']
results = []
seed = 42

print("Starting automated ablation sweep...")

for config in configs:
    print(f"\n--- Training config: {config} ---")
    # Run the training script
    subprocess.run([
        "python", "../train.py", 
        "--cov_config", config, 
        "--seed", str(seed),
        "--epochs", "15"
    ], check=True)
    
    # Read the saved metrics
    metrics_path = f"../results/metrics_{config}_seed_{seed}.json"
    if os.path.exists(metrics_path):
        with open(metrics_path, "r") as f:
            metrics = json.load(f)
            metrics['cov_config'] = config
            results.append(metrics)
    else:
        print(f"Warning: Could not find metrics for {config}")

# Compile and display the final leaderboard comparison
df_results = pd.DataFrame(results)
# Reorder columns for readability
df_results = df_results[['cov_config', 'rmse', 'mae', 'smape', 'epoch']]
df_results = df_results.sort_values('rmse')

display(df_results)
